### Loading Data

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = "../data/online_retail_II.xlsx"

sheets = pd.read_excel(DATA_PATH, sheet_name=None, engine="openpyxl")
list(sheets.keys())

['Year 2009-2010', 'Year 2010-2011']

### Do the sheets share a scheme and a date range?

In [3]:
for name, sheet_df in sheets.items():
    print(name, sheet_df.shape, sheet_df["InvoiceDate"].min(), sheet_df["InvoiceDate"].max())

sheets["Year 2009-2010"].columns.equals(sheets["Year 2010-2011"].columns)

Year 2009-2010 (525461, 8) 2009-12-01 07:45:00 2010-12-09 20:01:00
Year 2010-2011 (541910, 8) 2010-12-01 08:26:00 2011-12-09 12:50:00


True

### How many invoices identical rows or different transactions?

In [4]:
overlap = set(sheets["Year 2009-2010"]["Invoice"]) & set(sheets["Year 2010-2011"]["Invoice"])
print(f"{len(overlap)} invoice numbers appear in both sheets")

1088 invoice numbers appear in both sheets


### Are shared invoices identical rows or different transactions?

In [5]:
print(sheets["Year 2009-2010"][sheets["Year 2009-2010"]["Invoice"] == 536365])
print(sheets["Year 2010-2011"][sheets["Year 2010-2011"]["Invoice"] == 536365])

       Invoice StockCode                          Description  Quantity  \
502938  536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
502939  536365     71053                  WHITE METAL LANTERN         6   
502940  536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
502941  536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
502942  536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   
502943  536365     22752         SET 7 BABUSHKA NESTING BOXES         2   
502944  536365     21730    GLASS STAR FROSTED T-LIGHT HOLDER         6   

               InvoiceDate  Price  Customer ID         Country  
502938 2010-12-01 08:26:00   2.55      17850.0  United Kingdom  
502939 2010-12-01 08:26:00   3.39      17850.0  United Kingdom  
502940 2010-12-01 08:26:00   2.75      17850.0  United Kingdom  
502941 2010-12-01 08:26:00   3.39      17850.0  United Kingdom  
502942 2010-12-01 08:26:00   3.39      17850.0  United Kingdom  
502943 20

### How many duplicate rows would a naive concat produce?

In [6]:
combined = pd.concat(sheets.values(), ignore_index=True)
combined.duplicated().sum()

np.int64(34335)

### Where in time do the shared invoices sit?

In [7]:
overlap_in_2009 = sheets["Year 2009-2010"][sheets["Year 2009-2010"]["Invoice"].isin(overlap)]["InvoiceDate"]
overlap_in_2010 = sheets["Year 2010-2011"][sheets["Year 2010-2011"]["Invoice"].isin(overlap)]["InvoiceDate"]

print("In 2009-2010 sheet:", overlap_in_2009.min(), "to", overlap_in_2009.max())
print("In 2010-2011 sheet:", overlap_in_2010.min(), "to", overlap_in_2010.max())

In 2009-2010 sheet: 2010-12-01 08:26:00 to 2010-12-09 20:01:00
In 2010-2011 sheet: 2010-12-01 08:26:00 to 2010-12-09 20:01:00



### How many rows fall inside that window in each sheet?

In [8]:
window_start = overlap_in_2010.min()
window_end = overlap_in_2010.max()

rows_in_window_2009 = sheets["Year 2009-2010"]["InvoiceDate"].between(window_start, window_end).sum()
rows_in_window_2010 = sheets["Year 2010-2011"]["InvoiceDate"].between(window_start, window_end).sum()

print("Rows in window, 2009-2010 sheet:", rows_in_window_2009)
print("Rows in window, 2010-2011 sheet:", rows_in_window_2010)

Rows in window, 2009-2010 sheet: 22523
Rows in window, 2010-2011 sheet: 22523


### Is the second sheet's window a pure copy, and what duplicates exist inside each sheet?

In [9]:
window_2009 = sheets["Year 2009-2010"][sheets["Year 2009-2010"]["InvoiceDate"].between(window_start, window_end)]
window_2010 = sheets["Year 2010-2011"][sheets["Year 2010-2011"]["InvoiceDate"].between(window_start, window_end)]

print("Unique rows, 2009-2010 window:       ", len(window_2009.drop_duplicates()))
print("Unique rows, both windows combined:  ", len(pd.concat([window_2009, window_2010]).drop_duplicates()))
print("Exact duplicates inside 2009-2010 sheet:", sheets["Year 2009-2010"].duplicated().sum())
print("Exact duplicates inside 2010-2011 sheet:", sheets["Year 2010-2011"].duplicated().sum())

Unique rows, 2009-2010 window:        22202
Unique rows, both windows combined:   22202
Exact duplicates inside 2009-2010 sheet: 6865
Exact duplicates inside 2010-2011 sheet: 5268


### What do the within-sheet duplicates looks like?

In [10]:
cols = ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID"]

dups_2009 = sheets["Year 2009-2010"][sheets["Year 2009-2010"].duplicated(keep=False)]

print("Invoices affected:", dups_2009["Invoice"].nunique())
print(dups_2009[cols].head(10))

Invoices affected: 2538
    Invoice StockCode                        Description  Quantity  Price  \
362  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1   3.75   
363  489517     21912           VINTAGE SNAKES & LADDERS         1   3.75   
365  489517     21821   GLITTER STAR GARLAND WITH BELLS          1   3.75   
367  489517     22319  HAIRCLIPS FORTIES FABRIC ASSORTED        12   0.65   
368  489517     22130   PARTY CONE CHRISTMAS DECORATION          6   0.85   
371  489517     21912           VINTAGE SNAKES & LADDERS         1   3.75   
379  489517     21491    SET OF THREE VINTAGE GIFT WRAPS         1   1.95   
383  489517     22130   PARTY CONE CHRISTMAS DECORATION          6   0.85   
384  489517     22319  HAIRCLIPS FORTIES FABRIC ASSORTED        12   0.65   
385  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1   3.75   

     Customer ID  
362      16329.0  
363      16329.0  
365      16329.0  
367      16329.0  
368      16329.0  
371      16329

### How much value do the within-sheet duplicates carry?

In [11]:
for name, sheet_df in sheets.items():
    extra_copy = sheet_df.duplicated()
    line_total = sheet_df["Quantity"] * sheet_df["Price"]
    share = line_total[extra_copy].sum() / line_total.sum() * 100
    print(f"{name} | extra-copy rows: {extra_copy.sum()} | share of net value: {share:.2f}%")

Year 2009-2010 | extra-copy rows: 6865 | share of net value: 0.35%
Year 2010-2011 | extra-copy rows: 5268 | share of net value: 0.22%


### Observations

**Findings:** The two sheets aren't a clean split. They have the same columns, but the
date ranges overlap from 2010-12-01 08:26 to 2010-12-09 20:01. That window has 22,523
rows in each sheet and 22,202 unique, and putting the two windows together adds nothing
new, so the 2010-2011 sheet just repeats the end of 2009-2010. 1,088 invoice numbers
show up in both, and invoice 536365 has identical lines, prices and customer in each. A
plain concat gives 34,335 duplicate rows: 22,202 from the overlap, 6,865 inside
2009-2010 and 5,268 inside 2010-2011.

The 12,133 duplicates inside single sheets are a separate thing. In the first two
invoices I looked at, the repeated lines were spread through the invoice, not sitting
next to each other, with normal quantities (1, 6, 12). They're worth 0.35% and 0.22% of
net value.

**Decision:** Keep all of 2009-2010 and only take 2010-2011 rows after 2010-12-09 20:01.
That drops the overlap (22,523 rows) and nothing gets counted twice. I'm leaving the
within-sheet repeats in.

**Why:** The overlap is definitely an artefact, so dropping it was easy. The
within-sheet repeats are harder. There's no line number column, so I can't tell a
logging error from someone genuinely adding the same item twice. The sample looked more
like items added at different points in an order than a repeated log line, but that's
only two invoices, so I'm not claiming more than that. They're also tiny, under 0.4% of
value, so deleting them on a guess could remove real purchases for almost no gain.

**What I'm accepting:** If they are logging errors, Monetary is overstated by roughly
0.2 to 0.35% overall, and more for any customer with lots of repeated lines. Frequency
should be fine if it counts distinct invoices. I can check later by re-running the
clustering without the repeats and seeing if the segments move.

### Building df: both sheets, minus the overlap

In [13]:
# Last timestamp in 2009-2010. The 2010-2011 sheet repeats everything up to here,
# so only rows strictly after it are new.
OVERLAP_CUTOFF = sheets["Year 2009-2010"]["InvoiceDate"].max()

early = sheets["Year 2009-2010"]
late_new_rows = sheets["Year 2010-2011"][sheets["Year 2010-2011"]["InvoiceDate"] > OVERLAP_CUTOFF]

df = pd.concat([early, late_new_rows], ignore_index=True)

print(df.shape)
print(df.duplicated().sum())

(1044848, 8)
11812
